# Seguimiento ex-post: ¿cómo les fue a las corridas guardadas?

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/santiagoriverti/oportunidades_inversion/blob/main/notebooks/02_seguimiento_expost.ipynb)

Lee las corridas que guardó el notebook 01 en tu Google Drive (`Mi unidad/oportunidades_inversion/registros`) y,
con precios **actuales**, mide para cada una:

- cuánto rindió cada uno de los 8 portafolios desde la fecha de la corrida (comprando ese día y manteniendo),
  contra el benchmark (SPY), en total y a 1, 3, 6 y 12 meses;
- si las **señales** acertaron: cuánto rindieron las "Oportunidades" contra el resto y la correlación entre el
  score y el retorno posterior;
- un **ranking de estrategias** con todas las corridas juntas.

Si todavía no tenés corridas propias se usa la **corrida de ejemplo** del repo (lista por defecto, 06/11/2026).
Retorno anualizado, volatilidad y Sharpe aparecen recién con 3 meses de historia (con menos no significan nada).

> ⚠️ **Herramienta educativa y de análisis cuantitativo. No es asesoramiento financiero ni una recomendación de compra o venta.** Que un activo esté "barato" contra su historia no garantiza que vaya a subir, y los rendimientos pasados no garantizan rendimientos futuros.

In [ ]:
# Configuración: en Colab clona (o actualiza) el repo e instala la última yfinance; en la PC usa la carpeta del repo
import os, sys, subprocess, importlib
REPO = 'https://github.com/santiagoriverti/oportunidades_inversion.git'
if 'google.colab' in sys.modules:
    RAIZ = '/content/oportunidades_inversion'
    if os.path.exists(RAIZ):
        # Trae la última versión de GitHub y descarta cambios locales (la copia de Colab es descartable)
        subprocess.run(['git', '-C', RAIZ, 'fetch', '-q', '--depth', '1', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', RAIZ, 'reset', '-q', '--hard', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, RAIZ], check=True)
    # Yahoo cambia seguido: la última yfinance suele ser la que funciona
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'yfinance'], check=True)
else:  # subir desde la carpeta actual hasta encontrar el repo
    RAIZ = os.getcwd()
    while not os.path.exists(os.path.join(RAIZ, 'src', 'pipeline.py')) and os.path.dirname(RAIZ) != RAIZ:
        RAIZ = os.path.dirname(RAIZ)
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
# Si el notebook ya se corrió en esta sesión, olvidar los módulos de src cargados (pueden ser viejos)
for m in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
    del sys.modules[m]
importlib.invalidate_caches()
v = subprocess.run(['git', '-C', RAIZ, 'log', '-1', '--format=%h (%ad) %s', '--date=short'],
                   capture_output=True, text=True).stdout.strip()
print('Versión del repo:', v or 'sin git')

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import pandas as pd
from IPython.display import display, Markdown
from src.config import Config
from src.pipeline import Analisis
from src.portafolios import NOMBRES
from src import tablas, graficos, registro, expost, datos
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

In [ ]:
# ============================ CONFIGURACIÓN ============================
CARPETA = None          # None = Google Drive (Mi unidad/oportunidades_inversion/registros); o una ruta propia
INCLUIR_EJEMPLO = True  # suma la corrida de ejemplo del repo (data/ejemplo_registro)
BENCHMARK = "SPY"

carpeta = CARPETA or registro.carpeta_por_defecto()
lista = registro.listar_corridas(carpeta)
print(f"Corridas propias en {carpeta}: {len(lista)}")
rutas = list(lista["ruta"])
if INCLUIR_EJEMPLO or not rutas:
    ejemplo = registro.listar_corridas(os.path.join(RAIZ, "data", "ejemplo_registro"))
    rutas += [r for r in ejemplo["ruta"] if r not in rutas]
    lista = pd.concat([lista, ejemplo.assign(run_id=ejemplo["run_id"] + " (ejemplo)")], ignore_index=True)
display(lista.drop(columns=["ruta"]))
corridas = [registro.cargar_corrida(r) for r in rutas]

In [ ]:
R = None
if corridas:
    precios, avisos = expost.descargar_precios_corridas(corridas, BENCHMARK)
    for a in avisos:
        print("aviso:", a)
    rf = datos.tasa_libre_riesgo()[0]
    R = expost.evaluar_todas(corridas, precios, BENCHMARK, rf)
    print(f"Precios hasta el {precios.index.max().date()}; tasa libre de riesgo {rf:.2%}.")
else:
    print("No hay corridas para evaluar: corré primero el notebook 01 con GUARDAR_EN_DRIVE = True.")

## Ranking de estrategias (todas las corridas)

`exceso_medio` = retorno del portafolio menos el del benchmark en el mismo período, promediado entre corridas;
`pct_le_gana_al_benchmark` = en qué fracción de las corridas le ganó.

In [ ]:
if R is not None and len(R["ranking"]):
    display(tablas.estilo_expost(R["ranking"]))
    graficos.fig_ranking_expost(R["ranking"]).show()
    display(Markdown("### ¿Acertaron las señales?"))
    display(tablas.estilo_porcentajes(R["senales"], pct=["pct_oportunidades_ganan", "retorno_benchmark"],
                                      dec={"ic_score_vs_retorno": "{:+.2f}", "dias": "{:.0f}"}))

## Detalle de una corrida

Por defecto la más reciente (`CORRIDA = -1`); `0` es la más vieja.

In [ ]:
CORRIDA = -1
if R is not None and R["evaluaciones"]:
    e = R["evaluaciones"][CORRIDA]
    display(Markdown(f"### Corrida {e['run_id']}: datos al {e['fecha_datos'].date()} ({e['dias']} días hábiles)"))
    display(tablas.estilo_expost(e["tabla"].drop(columns=["clave"])))
    graficos.fig_curvas_expost(e["curvas"], f"Corrida {e['run_id']}: valor de 1 USD invertido").show()
    display(Markdown("**Retorno posterior por clasificación de la corrida:**"))
    display(tablas.estilo_porcentajes(e["por_clase"], pct=["retorno_medio", "retorno_mediano"]))
    graficos.fig_score_vs_retorno(e["senales"], e["retorno_benchmark"]).show()
    for a in e["avisos"]:
        print("aviso:", a)

## Descargar el seguimiento

ZIP con el Excel (ranking, todas las corridas, señales) y el reporte HTML con los gráficos.

In [ ]:
if R is not None:
    expost.exportar_seguimiento(R)